In [1]:
import pandas as pd
import joblib

# Load processed data
play_features = pd.read_csv(
    "../data/processed/play_level_features.csv"
)

# Load trained model
model = joblib.load(
    "../models/turfguard_logistic_model.pkl"
)

print("Model loaded successfully")

Model loaded successfully


In [5]:

# Exact feature lists from Notebook 07

numeric_features = [
    "Temperature",
    "tracking_points",
    "duration",
    "distance_total",
    "speed_mean",
    "speed_max",
    "speed_std",
    "acceleration_mean",
    "acceleration_abs_mean",
    "acceleration_abs_max",
    "x_range",
    "y_range"
]

categorical_features = [
    "FieldType",
    "StadiumType",
    "Weather",
    "PlayType",
    "Position",
    "PositionGroup",
    "RosterPosition",
    "temperature_category"
]

# Combine features in the original training order
feature_columns = numeric_features + categorical_features

# Check that every required feature exists
missing_features = [
    col for col in feature_columns
    if col not in sample.columns
]

if missing_features:
    raise ValueError(f"Missing features: {missing_features}")

# Prepare input
X_sample = sample[feature_columns].copy()

print("Total features:", len(feature_columns))
print("Input shape:", X_sample.shape)

display(X_sample)


Total features: 20
Input shape: (1, 20)


,Temperature,tracking_points,duration,distance_total,speed_mean,speed_max,speed_std,acceleration_mean,acceleration_abs_mean,acceleration_abs_max,x_range,y_range,FieldType,StadiumType,Weather,PlayType,Position,PositionGroup,RosterPosition,temperature_category
0,63,299.0,29.8,16.83,0.535753,2.94,0.607923,-0.000336,0.339262,2.7,7.13,6.23,synthetic,Outdoor,Clear and warm,Pass,QB,QB,Quarterback,Moderate


In [7]:
# Use the exact features used during model training
feature_columns = numeric_features + categorical_features

X_sample = sample[feature_columns].copy()

print("Features prepared:", X_sample.shape)

Features prepared: (1, 20)


In [8]:
prediction = model.predict(X_sample)[0]

probability = model.predict_proba(X_sample)[0][1]

print("Prediction:", prediction)
print("Recorded injury label score:",
      round(probability, 6))
print("Score percentage:",
      round(probability * 100, 4), "%")

Prediction: 0
Recorded injury label score: 9.1e-05
Score percentage: 0.0091 %


In [9]:
if prediction == 1:
    print("Model classified this play as a recorded-injury label.")
else:
    print("Model classified this play as no recorded-injury label.")

print("Model score:", round(probability, 6))

Model classified this play as no recorded-injury label.
Model score: 9.1e-05


In [11]:
# Load injury-enriched dataset from processed folder

injury_enriched = pd.read_csv(
    "../data/processed/injury_enriched_features.csv",
    dtype={"PlayKey": "string"}
)

# Ensure PlayKey has the same data type
play_features["PlayKey"] = play_features["PlayKey"].astype("string")

print("Injury data loaded:", injury_enriched.shape)
print("Injury records:", injury_enriched["PlayKey"].notna().sum())

Injury data loaded: (77, 41)
Injury records: 77


In [12]:
# Find positive and negative examples

injured_playkeys = set(
    injury_enriched["PlayKey"].dropna().unique()
)

test_data = play_features.copy()

test_data["actual_label"] = (
    test_data["PlayKey"].isin(injured_playkeys).astype(int)
)

# Select one recorded injury example
positive_sample = test_data[
    test_data["actual_label"] == 1
].iloc[[0]]

# Select one non-matched example
negative_sample = test_data[
    test_data["actual_label"] == 0
].iloc[[0]]

# Prepare both samples
X_positive = positive_sample[feature_columns]
X_negative = negative_sample[feature_columns]

# Generate scores
positive_score = model.predict_proba(X_positive)[0][1]
negative_score = model.predict_proba(X_negative)[0][1]

print("ACTUAL INJURY-LABEL EXAMPLE")
print("Actual label:", 1)
print("Predicted label:", model.predict(X_positive)[0])
print("Score:", positive_score)

print("\nNON-MATCHED EXAMPLE")
print("Actual label:", 0)
print("Predicted label:", model.predict(X_negative)[0])
print("Score:", negative_score)

ACTUAL INJURY-LABEL EXAMPLE
Actual label: 1
Predicted label: 1
Score: 0.639666620143753

NON-MATCHED EXAMPLE
Actual label: 0
Predicted label: 0
Score: 9.062818569405653e-05


In [14]:

from sklearn.model_selection import GroupShuffleSplit
import numpy as np
from sklearn.base import clone
from sklearn.metrics import (
    classification_report,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

# Reconstruct dataset and labels
test_data = play_features.copy()

test_data["recorded_injury_label"] = (
    test_data["PlayKey"]
    .isin(injured_playkeys)
    .astype(int)
)

X = test_data[feature_columns].copy()
y = test_data["recorded_injury_label"].copy()
groups = test_data["GameID"].copy()
X = X.replace([np.inf, -np.inf], np.nan)

# Split by GameID
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Training positives:", y_train.sum())
print("Testing positives:", y_test.sum())

# Create a fresh model with the same configuration
test_model = clone(model)

# Train only on training games
test_model.fit(X_train, y_train)

# Predict on unseen games
y_pred = test_model.predict(X_test)
y_prob = test_model.predict_proba(X_test)[:, 1]

# Evaluate
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

if y_test.nunique() == 2:
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print("PR-AUC:", average_precision_score(y_test, y_prob))
else:
    print("ROC-AUC and PR-AUC unavailable: test set has one class.")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Training records: 212898
Testing records: 54107
Training positives: 59
Testing positives: 17

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.81      0.89     54090
           1       0.00      0.41      0.00        17

    accuracy                           0.81     54107
   macro avg       0.50      0.61      0.45     54107
weighted avg       1.00      0.81      0.89     54107

ROC-AUC: 0.6259850140832817
PR-AUC: 0.000832681006024906

Confusion Matrix:
[[43616 10474]
 [   10     7]]


In [ ]:

from sklearn.model_selection import GroupShuffleSplit
from sklearn.base import clone
from sklearn.metrics import precision_score, recall_score, f1_score

# Split training data into training and validation sets
train_groups = groups.iloc[train_idx]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

sub_train_idx, val_idx = next(
    splitter.split(X_train, y_train, groups=train_groups)
)

X_subtrain = X_train.iloc[sub_train_idx]
y_subtrain = y_train.iloc[sub_train_idx]

X_val = X_train.iloc[val_idx]
y_val = y_train.iloc[val_idx]

# Train a fresh model
validation_model = clone(model)
validation_model.fit(X_subtrain, y_subtrain)

# Validation probabilities
val_prob = validation_model.predict_proba(X_val)[:, 1]

# Compare thresholds
results = []

for threshold in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
    val_pred = (val_prob >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, val_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, val_pred, zero_division=0
        ),
        "Predicted Positives": int(val_pred.sum())
    })

threshold_results = pd.DataFrame(results)

display(threshold_results)


,Threshold,Precision,Recall,F1,Predicted Positives
0,0.05,0.000211,0.444444,0.000421,18973
1,0.10,0.000125,0.222222,0.000250,16019
2,0.20,0.000159,0.222222,0.000318,12574
3,0.30,0.000194,0.222222,0.000388,10292
4,0.40,0.000117,0.111111,0.000235,8516
5,0.50,0.000143,0.111111,0.000285,7010


: 